# Tennis with Thompson

# USA Men's Tennis: 2026 Match-by-Match Analysis

This notebook builds analysis of **American men in ATP Masters 1000 and Grand Slam main draws during 2026**. It downloads TennisMyLife's completed-season and ongoing-tournament CSV files:

**Data source:** [TennisMyLife Tennis Match Database](https://stats.tennismylife.org/tennis-match-database).

## 1. Setup and configuration

NOTE: Change `SELECTED_PLAYER` later (in section 8) to drill into a different American player.

In [2]:
from pathlib import Path
from datetime import datetime, timezone
import io
import re
import sys
import shlex
import platform
import subprocess
import warnings

import numpy as np
import pandas as pd
import altair as alt

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
alt.data_transformers.disable_max_rows()

DATA_DIR = Path("data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

URLS = {
    "completed": "https://stats.tennismylife.org/data/2026.csv",
    "ongoing": "https://stats.tennismylife.org/data/ongoing_tourneys.csv",
}

RELEVANT_LEVELS = {"G", "M"}  # Grand Slam and Masters 1000 in the actual 2026 files
EXPECTED_EVENTS = {
    "Australian Open", "Indian Wells Masters", "Miami Masters", "Monte Carlo",
    "Madrid Masters", "Rome Masters", "Roland Garros", "Wimbledon",
    "Canada Masters", "Cincinnati Masters", "US Open", "Shanghai Masters",
    "Paris Masters",
}

REFRESH_DATA = True
print(f"Congrats dude, the notebook ran")

Congrats dude, the notebook ran


## 2. Download completed and ongoing data

In [4]:
def download_csv(label, url, refresh=True):
    cache_path = RAW_DIR / f"{label}_2026.csv"
    if refresh or not cache_path.exists():
        try:
            request = Request(url, headers={"User-Agent": "Mozilla/5.0 tennis-analysis-notebook"})
            with urlopen(request, timeout=45) as response:
                content = response.read()
            frame = pd.read_csv(io.BytesIO(content))
            frame.to_csv(cache_path, index=False)
            print(f"Downloaded {label}: {len(frame):,} rows")
            return frame
        except Exception as exc:
            if not cache_path.exists():
                raise RuntimeError(f"Could not download {label}, and no cache exists") from exc
            print(f"Download failed for {label}; using cached file ({exc})")
    frame = pd.read_csv(cache_path)
    print(f"Loaded cached {label}: {len(frame):,} rows")
    return frame


completed_raw = download_csv("completed", URLS["completed"], REFRESH_DATA)
ongoing_raw = download_csv("ongoing", URLS["ongoing"], REFRESH_DATA)
display(completed_raw.head(3))

Download failed for completed; using cached file (name 'Request' is not defined)
Loaded cached completed: 1,990 rows
Download failed for ongoing; using cached file (name 'Request' is not defined)
Loaded cached ongoing: 65 rows


,tourney_id,tourney_name,surface,draw_size,tourney_level,indoor,tourney_date,match_num,winner_id,winner_seed,winner_entry,winner_name,winner_hand,winner_ht,winner_ioc,winner_age,winner_rank,winner_rank_points,loser_id,loser_seed,loser_entry,loser_name,loser_hand,loser_ht,loser_ioc,loser_age,loser_rank,loser_rank_points,score,best_of,round,minutes,w_ace,w_df,w_svpt,w_1stIn,w_1stWon,w_2ndWon,w_SvGms,w_bpSaved,w_bpFaced,l_ace,l_df,l_svpt,l_1stIn,l_1stWon,l_2ndWon,l_SvGms,l_bpSaved,l_bpFaced
0,2026-9900,United Cup,Hard,18,A,O,20260102,1,B0BI,NaN,NaN,Sebastian Baez,R,170.0,ARG,25.002,45.0,1145.0,MU94,NaN,NaN,Jaume Munar,R,183.0,ESP,28.652,33.0,1385.0,6-4 6-4,3,RR,103.0,0.0,2.0,71.0,46.0,27.0,15.0,10.0,4.0,6.0,7.0,0.0,52.0,39.0,25.0,6.0,10.0,2.0,6.0
1,2026-9900,United Cup,Hard,18,A,O,20260102,2,TE51,NaN,NaN,Stefanos Tsitsipas,R,193.0,GRE,27.381,36.0,1380.0,M0HU,NaN,NaN,Shintaro Mochizuki,R,175.0,JPN,22.576,99.0,647.0,6-3 6-4,3,RR,82.0,8.0,2.0,61.0,42.0,28.0,11.0,10.0,1.0,2.0,1.0,3.0,57.0,30.0,22.0,11.0,9.0,5.0,8.0
2,2026-9900,United Cup,Hard,18,A,O,20260103,3,Z371,NaN,NaN,Zhizhen Zhang,R,193.0,CHN,29.202,410.0,115.0,BU13,NaN,NaN,Zizou Bergs,R,185.0,BEL,26.574,42.0,1208.0,6-7(2) 7-6(3) 7-5,3,RR,173.0,12.0,0.0,108.0,71.0,59.0,21.0,18.0,0.0,0.0,18.0,5.0,126.0,75.0,58.0,29.0,18.0,9.0,10.0


## 3. Clean, validate, and deduplicate

Live records can overlap completed records. The pipeline prefers the ongoing version when keys overlap, then keeps the most complete record. The primary key is tournament ID plus match number; a fallback composite key handles missing match numbers.

In [6]:
REQUIRED_COLUMNS = {
    "tourney_id", "tourney_name", "surface", "tourney_level", "tourney_date",
    "match_num", "winner_id", "winner_name", "winner_ioc", "winner_rank",
    "loser_id", "loser_name", "loser_ioc", "loser_rank", "score", "round",
    "minutes", "w_ace", "w_df", "w_svpt", "w_1stIn", "w_1stWon", "w_2ndWon",
    "w_bpSaved", "w_bpFaced", "l_ace", "l_df", "l_svpt", "l_1stIn",
    "l_1stWon", "l_2ndWon", "l_bpSaved", "l_bpFaced"
}

def validate_schema(frame, label):
    missing = sorted(REQUIRED_COLUMNS - set(frame.columns))
    if missing:
        raise ValueError(f"{label} is missing required columns: {missing}")

validate_schema(completed_raw, "completed")
validate_schema(ongoing_raw, "ongoing")

completed = completed_raw.assign(data_source="completed", source_priority=1)
ongoing = ongoing_raw.assign(data_source="ongoing", source_priority=2)
matches = pd.concat([completed, ongoing], ignore_index=True)

matches["match_date"] = pd.to_datetime(
    matches["tourney_date"].astype("Int64").astype(str), format="%Y%m%d", errors="coerce"
)
matches["year"] = matches["match_date"].dt.year
matches["row_completeness"] = matches.notna().sum(axis=1)
matches["fallback_key"] = (
    matches["tourney_id"].astype(str) + "|" + matches["round"].astype(str) + "|" +
    matches["winner_id"].astype(str) + "|" + matches["loser_id"].astype(str)
)
matches["match_key"] = np.where(
    matches["match_num"].notna(),
    matches["tourney_id"].astype(str) + "|" + matches["match_num"].astype(str),
    matches["fallback_key"]
)

rows_before = len(matches)
matches = (matches.sort_values(["source_priority", "row_completeness"])
           .drop_duplicates("match_key", keep="last")
           .reset_index(drop=True))
duplicates_removed = rows_before - len(matches)

# Restrict to the 2026 men's main-draw events requested.
big_events = matches[
    matches["year"].eq(2026)
    & matches["tourney_level"].isin(RELEVANT_LEVELS)
    & matches["tourney_name"].isin(EXPECTED_EVENTS)
].copy()

validation = pd.DataFrame({
    "check": ["Combined raw rows", "Duplicate rows removed", "Rows after deduplication",
              "Big-event matches", "Missing match dates", "Unexpected selected events"],
    "value": [rows_before, duplicates_removed, len(matches), len(big_events),
              int(big_events["match_date"].isna().sum()),
              int((~big_events["tourney_name"].isin(EXPECTED_EVENTS)).sum())]
})
display(validation)
display(big_events[["tourney_id", "tourney_name", "tourney_level", "surface"]]
        .drop_duplicates().sort_values("tourney_id").reset_index(drop=True))

,check,value
0,Combined raw rows,2055
1,Duplicate rows removed,1
2,Rows after deduplication,2054
3,Big-event matches,975
4,Missing match dates,0
5,Unexpected selected events,0


,tourney_id,tourney_name,tourney_level,surface
0,2026-1536,Madrid Masters,M,Clay
1,2026-403,Miami Masters,M,Hard
2,2026-404,Indian Wells Masters,M,Hard
3,2026-410,Monte Carlo,M,Clay
4,2026-416,Rome Masters,M,Clay
5,2026-421,Canada Masters,M,Hard
6,2026-422,Cincinnati Masters,M,Hard
7,2026-520,Roland Garros,G,Clay
8,2026-540,Wimbledon,G,Grass
9,2026-580,Australian Open,G,Hard


## 4. Create the player-centric U.S. dataset

One row represents one American player's perspective on one match. An all-American match correctly produces two analytical rows. Break-point conversions are inferred from the opponent's break points faced minus break points saved.

In [8]:
def safe_divide(numerator, denominator):
    numerator = pd.to_numeric(numerator, errors="coerce")
    denominator = pd.to_numeric(denominator, errors="coerce")
    return numerator.div(denominator.where(denominator.ne(0)))


def player_perspective(frame, side):
    is_winner = side == "winner"
    p = "winner" if is_winner else "loser"
    o = "loser" if is_winner else "winner"
    ps = "w" if is_winner else "l"
    os = "l" if is_winner else "w"
    eligible = frame[f"{p}_ioc"].eq("USA")
    d = frame.loc[eligible].copy()

    out = pd.DataFrame({
        "match_key": d["match_key"], "match_date": d["match_date"], "year": d["year"],
        "tourney_id": d["tourney_id"], "tournament": d["tourney_name"],
        "event_type": np.where(d["tourney_level"].eq("G"), "Grand Slam", "Masters 1000"),
        "surface": d["surface"], "indoor": d["indoor"], "round": d["round"],
        "best_of": d["best_of"], "score": d["score"], "minutes": d["minutes"],
        "data_source": d["data_source"], "player_id": d[f"{p}_id"],
        "player": d[f"{p}_name"], "player_rank": d[f"{p}_rank"],
        "player_seed": d[f"{p}_seed"], "player_entry": d[f"{p}_entry"],
        "opponent_id": d[f"{o}_id"], "opponent": d[f"{o}_name"],
        "opponent_country": d[f"{o}_ioc"], "opponent_rank": d[f"{o}_rank"],
        "opponent_seed": d[f"{o}_seed"], "result": "W" if is_winner else "L",
        "win": int(is_winner), "aces": d[f"{ps}_ace"], "double_faults": d[f"{ps}_df"],
        "serve_points": d[f"{ps}_svpt"], "first_serves_in": d[f"{ps}_1stIn"],
        "first_serve_points_won": d[f"{ps}_1stWon"],
        "second_serve_points_won": d[f"{ps}_2ndWon"],
        "service_games": d[f"{ps}_SvGms"], "bp_saved": d[f"{ps}_bpSaved"],
        "bp_faced": d[f"{ps}_bpFaced"], "opp_bp_saved": d[f"{os}_bpSaved"],
        "opp_bp_faced": d[f"{os}_bpFaced"],
    })
    return out


usa = pd.concat([
    player_perspective(big_events, "winner"),
    player_perspective(big_events, "loser")
], ignore_index=True)

numeric_cols = [
    "player_rank", "opponent_rank", "minutes", "aces", "double_faults", "serve_points",
    "first_serves_in", "first_serve_points_won", "second_serve_points_won",
    "service_games", "bp_saved", "bp_faced", "opp_bp_saved", "opp_bp_faced"
]
usa[numeric_cols] = usa[numeric_cols].apply(pd.to_numeric, errors="coerce")

usa["ranking_difference"] = usa["opponent_rank"] - usa["player_rank"]
usa["upset_win"] = ((usa["win"] == 1) & (usa["player_rank"] > usa["opponent_rank"])).astype(int)
usa["upset_loss"] = ((usa["win"] == 0) & (usa["player_rank"] < usa["opponent_rank"])).astype(int)
usa["first_serve_in_pct"] = safe_divide(usa["first_serves_in"], usa["serve_points"])
usa["first_serve_win_pct"] = safe_divide(usa["first_serve_points_won"], usa["first_serves_in"])
second_serve_points = usa["serve_points"] - usa["first_serves_in"]
usa["second_serve_win_pct"] = safe_divide(usa["second_serve_points_won"], second_serve_points)
usa["bp_save_pct"] = safe_divide(usa["bp_saved"], usa["bp_faced"])
usa["bp_opportunities"] = usa["opp_bp_faced"]
usa["bp_converted"] = usa["opp_bp_faced"] - usa["opp_bp_saved"]
usa["bp_conversion_pct"] = safe_divide(usa["bp_converted"], usa["bp_opportunities"])
usa["ace_rate"] = safe_divide(usa["aces"], usa["serve_points"])
usa["double_fault_rate"] = safe_divide(usa["double_faults"], usa["serve_points"])

usa = usa.sort_values(["match_date", "tournament", "match_key", "player"]).reset_index(drop=True)
output_path = PROCESSED_DIR / "usa_men_big_events_2026.csv"
usa.to_csv(output_path, index=False)

print(f"Analytical rows: {len(usa):,}")
print(f"Unique matches involving a U.S. player: {usa['match_key'].nunique():,}")
print(f"American players: {usa['player'].nunique():,}")
print(f"Saved: {output_path}")
display(usa.head())

Analytical rows: 275
Unique matches involving a U.S. player: 254
American players: 24
Saved: data/processed/usa_men_big_events_2026.csv


,match_key,match_date,year,tourney_id,tournament,event_type,surface,indoor,round,best_of,score,minutes,data_source,player_id,player,player_rank,player_seed,player_entry,opponent_id,opponent,opponent_country,opponent_rank,opponent_seed,result,win,aces,double_faults,serve_points,first_serves_in,first_serve_points_won,second_serve_points_won,service_games,bp_saved,bp_faced,opp_bp_saved,opp_bp_faced,ranking_difference,upset_win,upset_loss,first_serve_in_pct,first_serve_win_pct,second_serve_win_pct,bp_save_pct,bp_opportunities,bp_converted,bp_conversion_pct,ace_rate,double_fault_rate
0,2026-580|12,2026-01-18,2026,2026-580,Australian Open,Grand Slam,Hard,O,R128,5,6-4 6-4 3-6 6-7(0) 6-3,224.0,completed,Z0BZ,Michael Zheng,174.0,NaN,NaN,K0AH,Sebastian Korda,USA,53.0,NaN,W,1,4.0,0.0,174.0,115.0,79.0,34.0,0.0,6.0,7.0,4.0,8.0,-121.0,1,0,0.660920,0.686957,0.576271,0.857143,8.0,4.0,0.500000,0.022989,0.000000
1,2026-580|12,2026-01-18,2026,2026-580,Australian Open,Grand Slam,Hard,O,R128,5,6-4 6-4 3-6 6-7(0) 6-3,224.0,completed,K0AH,Sebastian Korda,53.0,NaN,NaN,Z0BZ,Michael Zheng,USA,174.0,NaN,L,0,22.0,5.0,142.0,89.0,71.0,31.0,0.0,4.0,8.0,6.0,7.0,121.0,0,1,0.626761,0.797753,0.584906,0.500000,7.0,1.0,0.142857,0.154930,0.035211
2,2026-580|13,2026-01-18,2026,2026-580,Australian Open,Grand Slam,Hard,O,R128,5,6-4 6-4 6-4,127.0,completed,B0CD,Jenson Brooksby,48.0,NaN,NaN,BK92,Alexander Bublik,KAZ,10.0,10.0,L,0,4.0,1.0,97.0,61.0,38.0,20.0,0.0,7.0,11.0,7.0,8.0,-38.0,0,0,0.628866,0.622951,0.555556,0.636364,8.0,1.0,0.125000,0.041237,0.010309
3,2026-580|14,2026-01-18,2026,2026-580,Australian Open,Grand Slam,Hard,O,R128,5,7-5 4-6 6-4 7-6(3),172.0,completed,S0K7,Zachary Svajda,143.0,NaN,NaN,H997,Yannick Hanfmann,GER,102.0,NaN,L,0,11.0,8.0,143.0,102.0,69.0,18.0,0.0,4.0,9.0,4.0,8.0,-41.0,0,0,0.713287,0.676471,0.439024,0.444444,8.0,4.0,0.500000,0.076923,0.055944
4,2026-580|6,2026-01-18,2026,2026-580,Australian Open,Grand Slam,Hard,O,R128,5,6-2 6-3 3-6 6-3,168.0,completed,K0A3,Patrick Kypson,116.0,NaN,NaN,C0DF,Francisco Comesana,ARG,68.0,NaN,L,0,12.0,7.0,118.0,111.0,68.0,0.0,0.0,6.0,10.0,1.0,2.0,-48.0,0,0,0.940678,0.612613,0.000000,0.600000,2.0,1.0,0.500000,0.101695,0.059322


## 5. CLUTH TIME: deciding sets and tiebreaks

btw...it is called a tiebreak, not a tiebreaker (pet peeve)

In [10]:
SET_PATTERN = re.compile(r"(?<!\d)(\d+)-(\d+)(?:\((\d+)\))?")

def parse_winner_score(score):
    text = "" if pd.isna(score) else str(score).strip().upper()
    sets = [(int(a), int(b), tb) for a, b, tb in SET_PATTERN.findall(text)]
    winner_sets = sum(a > b for a, b, _ in sets)
    loser_sets = sum(b > a for a, b, _ in sets)
    tiebreak_sets = [(a, b) for a, b, tb in sets if tb != ""]
    winner_tiebreaks = sum(a > b for a, b in tiebreak_sets)
    loser_tiebreaks = sum(b > a for a, b in tiebreak_sets)
    return pd.Series({
        "completed_sets": winner_sets + loser_sets,
        "winner_sets_won": winner_sets,
        "loser_sets_won": loser_sets,
        "tiebreaks_played": len(tiebreak_sets),
        "winner_tiebreaks_won": winner_tiebreaks,
        "loser_tiebreaks_won": loser_tiebreaks,
        "winner_won_first_set": bool(sets and sets[0][0] > sets[0][1]),
        "retirement_or_abandoned": any(flag in text for flag in ["RET", "ABD"]),
        "walkover_or_default": any(flag in text for flag in ["W/O", "WO", "DEF"]),
        "score_parseable": len(sets) > 0,
    })

score_features = big_events[["match_key", "best_of", "score"]].copy()
score_features = pd.concat(
    [score_features, score_features["score"].apply(parse_winner_score)], axis=1
)
score_features["deciding_set_played"] = (
    score_features["score_parseable"]
    & score_features["completed_sets"].eq(pd.to_numeric(score_features["best_of"], errors="coerce"))
)
score_features["five_set_match"] = (
    pd.to_numeric(score_features["best_of"], errors="coerce").eq(5)
    & score_features["completed_sets"].eq(5)
)

usa = usa.merge(
    score_features.drop(columns=["best_of", "score"]),
    on="match_key", how="left", validate="many_to_one"
)
usa["player_sets_won"] = np.where(usa["win"].eq(1), usa["winner_sets_won"], usa["loser_sets_won"])
usa["opponent_sets_won"] = np.where(usa["win"].eq(1), usa["loser_sets_won"], usa["winner_sets_won"])
usa["player_tiebreaks_won"] = np.where(
    usa["win"].eq(1), usa["winner_tiebreaks_won"], usa["loser_tiebreaks_won"]
)
usa["tiebreak_win_pct"] = safe_divide(usa["player_tiebreaks_won"], usa["tiebreaks_played"])
usa["straight_set_win"] = (
    usa["win"].eq(1) & usa["score_parseable"] & usa["opponent_sets_won"].eq(0)
).astype(int)
usa["deciding_set_win"] = (usa["deciding_set_played"] & usa["win"].eq(1)).astype(int)
usa["five_set_win"] = (usa["five_set_match"] & usa["win"].eq(1)).astype(int)
usa["comeback_win"] = (
    usa["win"].eq(1) & usa["score_parseable"] & ~usa["winner_won_first_set"]
).astype(int)

score_quality = pd.DataFrame({
    "metric": ["Rows with parseable scores", "Deciding-set rows", "Five-set rows",
               "Rows with a tiebreak", "Retirement/abandoned rows", "Walkover/default rows"],
    "value": [usa["score_parseable"].sum(), usa["deciding_set_played"].sum(),
              usa["five_set_match"].sum(), usa["tiebreaks_played"].gt(0).sum(),
              usa["retirement_or_abandoned"].sum(), usa["walkover_or_default"].sum()]
})
display(score_quality)

,metric,value
0,Rows with parseable scores,275
1,Deciding-set rows,82
2,Five-set rows,24
3,Rows with a tiebreak,133
4,Retirement/abandoned rows,6
5,Walkover/default rows,0


## 6. Ranking-based expectations

The player's estimated probability is `opponent_rank / (player_rank + opponent_rank)`. It uses only the two match-time rankings, not the match result. This is an interpretable benchmark. It is not a fully calibrated forecasting model.

- **Expected win:** probability at least 65%
- **Toss-up:** probability between 35% and 65%
- **Expected loss:** probability at most 35%
- **Performance versus expectation:** actual result (1/0) minus expected probability

In [12]:
valid_ranks = usa["player_rank"].gt(0) & usa["opponent_rank"].gt(0)
usa["ranking_expected_win_prob"] = np.where(
    valid_ranks,
    usa["opponent_rank"] / (usa["player_rank"] + usa["opponent_rank"]),
    np.nan,
)
usa["expectation_group"] = pd.cut(
    usa["ranking_expected_win_prob"],
    bins=[-np.inf, 0.35, 0.65, np.inf],
    labels=["Expected loss", "Toss-up", "Expected win"],
    right=False,
)
usa["performance_vs_expectation"] = usa["win"] - usa["ranking_expected_win_prob"]
usa["expectation_outcome"] = np.select(
    [
        usa["win"].eq(1) & usa["expectation_group"].eq("Expected loss"),
        usa["win"].eq(0) & usa["expectation_group"].eq("Expected win"),
        usa["win"].eq(1),
    ],
    ["Upset win", "Unexpected loss", "Win"],
    default="Loss",
)

expectation_summary = (usa.groupby("player", as_index=False, observed=True)
    .agg(matches=("win", "size"), actual_win_pct=("win", "mean"),
         expected_win_pct=("ranking_expected_win_prob", "mean"),
         performance_vs_expectation=("performance_vs_expectation", "mean"),
         upset_wins=("expectation_outcome", lambda s: s.eq("Upset win").sum()),
         unexpected_losses=("expectation_outcome", lambda s: s.eq("Unexpected loss").sum()))
    .sort_values("performance_vs_expectation", ascending=False))

display(expectation_summary.round(3))

# Re-export after adding score and expectation features.
usa.to_csv(output_path, index=False)
print(f"Updated analytical dataset: {output_path}")

,player,matches,actual_win_pct,expected_win_pct,performance_vs_expectation,upset_wins,unexpected_losses
4,Darwin Blanch,2,0.500,0.163,0.337,1,0
16,Nishesh Basavareddy,4,0.500,0.169,0.331,2,0
15,Michael Zheng,10,0.500,0.255,0.245,3,0
5,Eliot Spizzirri,4,0.500,0.289,0.211,1,0
23,Zachary Svajda,16,0.500,0.358,0.142,3,1
14,Martin Damm,9,0.333,0.233,0.100,1,0
1,Alex Michelsen,21,0.571,0.493,0.079,1,3
11,Learner Tien,25,0.680,0.612,0.068,3,3
8,Frances Tiafoe,22,0.682,0.636,0.045,0,2
18,Reilly Opelka,9,0.444,0.413,0.032,1,2


Updated analytical dataset: data/processed/usa_men_big_events_2026.csv


## 7. Player summary table

In [14]:
summary = (usa.groupby("player", as_index=False)
           .agg(matches=("match_key", "count"), wins=("win", "sum"),
                win_pct=("win", "mean"), tournaments=("tournament", "nunique"),
                best_rank=("player_rank", "min"), upsets=("upset_win", "sum"),
                avg_aces=("aces", "mean"), first_serve_in_pct=("first_serve_in_pct", "mean"),
                first_serve_win_pct=("first_serve_win_pct", "mean"),
                second_serve_win_pct=("second_serve_win_pct", "mean"),
                bp_save_pct=("bp_save_pct", "mean"),
                bp_conversion_pct=("bp_conversion_pct", "mean"),
                deciding_sets=("deciding_set_played", "sum"),
                deciding_set_wins=("deciding_set_win", "sum"),
                tiebreaks=("tiebreaks_played", "sum"),
                tiebreaks_won=("player_tiebreaks_won", "sum"),
                comeback_wins=("comeback_win", "sum"),
                expected_win_pct=("ranking_expected_win_prob", "mean"),
                performance_vs_expectation=("performance_vs_expectation", "mean"))
           .assign(losses=lambda x: x["matches"] - x["wins"])
           .sort_values(["wins", "win_pct", "matches"], ascending=False))

pct_cols = ["win_pct", "first_serve_in_pct", "first_serve_win_pct",
            "second_serve_win_pct", "bp_save_pct", "bp_conversion_pct",
            "expected_win_pct", "performance_vs_expectation"]
summary_display = summary.copy()
summary_display[pct_cols] = (summary_display[pct_cols] * 100).round(1)
display(summary_display.rename(columns={c: f"{c} (%)" for c in pct_cols}))

,player,matches,wins,win_pct (%),tournaments,best_rank,upsets,avg_aces,first_serve_in_pct (%),first_serve_win_pct (%),second_serve_win_pct (%),bp_save_pct (%),bp_conversion_pct (%),deciding_sets,deciding_set_wins,tiebreaks,tiebreaks_won,comeback_wins,expected_win_pct (%),performance_vs_expectation (%),losses
11,Learner Tien,25,17,68.0,9,12.0,4,6.440000,65.3,66.5,51.9,54.2,43.3,7,6,14,11,2,61.2,6.8,8
8,Frances Tiafoe,22,15,68.2,8,19.0,2,8.136364,59.0,73.9,52.7,69.2,42.2,8,6,18,8,5,63.6,4.5,7
21,Tommy Paul,22,14,63.6,9,18.0,1,6.590909,61.2,75.5,59.0,58.7,45.1,6,3,11,5,1,66.8,-3.2,8
2,Ben Shelton,20,12,60.0,9,5.0,0,9.800000,68.0,79.1,58.7,71.4,34.4,6,1,15,10,2,81.2,-21.2,8
1,Alex Michelsen,21,12,57.1,9,37.0,4,6.380952,65.4,74.2,50.1,63.5,40.4,8,4,9,5,1,49.3,7.9,9
20,Taylor Fritz,17,11,64.7,7,7.0,0,15.235294,66.8,78.5,55.5,62.7,29.2,3,2,12,7,1,81.5,-16.8,6
3,Brandon Nakashima,19,11,57.9,9,22.0,3,11.421053,67.8,79.9,52.8,60.8,41.9,7,3,16,4,2,60.2,-2.3,8
23,Zachary Svajda,16,8,50.0,8,66.0,6,8.500000,62.6,72.3,49.1,54.4,38.9,6,4,11,4,4,35.8,14.2,8
15,Michael Zheng,10,5,50.0,6,110.0,4,7.400000,63.3,67.5,47.0,61.6,38.8,4,3,10,3,3,25.5,24.5,5
7,Ethan Quinn,13,5,38.5,8,47.0,4,8.307692,67.9,73.4,50.2,62.2,30.7,1,0,12,3,0,44.1,-5.7,8


## 8. Match-by-match drill-down

Choose any name from `sorted(usa.player.unique())`. The table includes the requested opponent, result, event context, rankings, serve statistics, break-point measures, duration, and score.

In [16]:
print(sorted(usa["player"].unique()))
SELECTED_PLAYER = "Tommy Paul"

detail_columns = [
    "match_date", "player", "opponent", "opponent_country", "result", "tournament",
    "event_type", "surface", "round", "player_rank", "opponent_rank",
    "ranking_difference", "upset_win", "aces", "double_faults", "first_serve_in_pct",
    "first_serve_win_pct", "second_serve_win_pct", "bp_save_pct", "bp_converted",
    "bp_opportunities", "bp_conversion_pct", "player_sets_won", "opponent_sets_won",
    "tiebreaks_played", "player_tiebreaks_won", "deciding_set_played", "comeback_win",
    "ranking_expected_win_prob", "expectation_group", "performance_vs_expectation",
    "minutes", "score", "data_source"
]
player_matches = usa.loc[usa["player"].eq(SELECTED_PLAYER), detail_columns]
player_matches_display = player_matches.sort_values("match_date", ascending=False).copy()
detail_pct_cols = ["first_serve_in_pct", "first_serve_win_pct", "second_serve_win_pct",
                   "bp_save_pct", "bp_conversion_pct", "ranking_expected_win_prob",
                   "performance_vs_expectation"]
player_matches_display[detail_pct_cols] = (player_matches_display[detail_pct_cols] * 100).round(1)
display(player_matches_display.rename(columns={c: f"{c} (%)" for c in detail_pct_cols}))

['Aleksandar Kovacevic', 'Alex Michelsen', 'Ben Shelton', 'Brandon Nakashima', 'Darwin Blanch', 'Eliot Spizzirri', 'Emilio Nava', 'Ethan Quinn', 'Frances Tiafoe', 'J.J. Wolf', 'Jenson Brooksby', 'Learner Tien', 'Mackenzie McDonald', 'Marcos Giron', 'Martin Damm', 'Michael Zheng', 'Nishesh Basavareddy', 'Patrick Kypson', 'Reilly Opelka', 'Sebastian Korda', 'Taylor Fritz', 'Tommy Paul', 'Tristan Boyer', 'Zachary Svajda']


,match_date,player,opponent,opponent_country,result,tournament,event_type,surface,round,player_rank,opponent_rank,ranking_difference,upset_win,aces,double_faults,first_serve_in_pct (%),first_serve_win_pct (%),second_serve_win_pct (%),bp_save_pct (%),bp_converted,bp_opportunities,bp_conversion_pct (%),player_sets_won,opponent_sets_won,tiebreaks_played,player_tiebreaks_won,deciding_set_played,comeback_win,ranking_expected_win_prob (%),expectation_group,performance_vs_expectation (%),minutes,score,data_source
266,2026-08-16,Tommy Paul,Hubert Hurkacz,POL,W,Cincinnati Masters,Masters 1000,Hard,R64,24.0,69.0,45.0,0,1.0,3.0,57.3,73.0,59.6,100.0,2.0,6.0,33.3,2,1,1,0,True,0,74.2,Expected win,25.8,157.0,6-4 6-7(3) 6-3,ongoing
248,2026-08-08,Tommy Paul,Learner Tien,USA,L,Canada Masters,Masters 1000,Hard,R32,21.0,19.0,-2.0,0,4.0,1.0,59.3,56.2,40.9,50.0,2.0,5.0,40.0,0,2,0,0,False,0,47.5,Toss-up,-47.5,78.0,6-3 6-2,completed
242,2026-08-06,Tommy Paul,Valentin Royer,FRA,W,Canada Masters,Masters 1000,Hard,R64,21.0,203.0,182.0,0,7.0,4.0,60.7,74.1,48.6,88.9,2.0,5.0,40.0,2,0,1,1,False,0,90.6,Expected win,9.4,124.0,6-4 7-6(5),completed
219,2026-07-03,Tommy Paul,Hubert Hurkacz,POL,L,Wimbledon,Grand Slam,Grass,R32,25.0,96.0,71.0,0,6.0,6.0,58.0,71.2,58.6,57.1,1.0,5.0,20.0,1,3,1,0,False,0,79.3,Expected win,-79.3,190.0,4-6 7-6(5) 7-5 6-2,completed
206,2026-07-01,Tommy Paul,Soonwoo Kwon,KOR,W,Wimbledon,Grand Slam,Grass,R64,25.0,200.0,175.0,0,19.0,1.0,64.5,78.3,60.6,0.0,4.0,8.0,50.0,3,0,1,1,False,0,88.9,Expected win,11.1,131.0,6-3 7-6(4) 6-2,completed
194,2026-06-29,Tommy Paul,Alexandre Muller,FRA,W,Wimbledon,Grand Slam,Grass,R128,25.0,126.0,101.0,0,12.0,1.0,69.0,85.0,66.7,NaN,7.0,16.0,43.8,3,0,0,0,False,0,83.4,Expected win,16.6,72.0,6-1 6-2 6-1,completed
178,2026-05-29,Tommy Paul,Casper Ruud,NOR,L,Roland Garros,Grand Slam,Clay,R32,21.0,16.0,-5.0,0,14.0,5.0,59.0,78.9,57.6,0.0,2.0,14.0,14.3,2,3,2,1,True,0,43.2,Toss-up,-43.2,287.0,4-6 6-7(4) 6-4 7-6(4) 7-5,completed
174,2026-05-27,Tommy Paul,Lorenzo Sonego,ITA,W,Roland Garros,Grand Slam,Clay,R64,21.0,70.0,49.0,0,2.0,2.0,58.0,78.4,48.6,83.3,5.0,8.0,62.5,3,0,0,0,False,0,76.9,Expected win,23.1,130.0,6-3 6-2 6-4,completed
167,2026-05-25,Tommy Paul,Rinky Hijikata,AUS,W,Roland Garros,Grand Slam,Clay,R128,21.0,98.0,77.0,0,13.0,3.0,58.6,67.1,62.1,75.0,5.0,11.0,45.5,3,1,0,0,False,1,82.4,Expected win,17.6,200.0,4-6 6-3 7-5 6-4,completed
151,2026-05-10,Tommy Paul,Luciano Darderi,ITA,L,Rome Masters,Masters 1000,Clay,R32,18.0,20.0,2.0,0,5.0,3.0,55.9,55.8,56.1,55.6,2.0,4.0,50.0,1,2,0,0,True,0,52.6,Toss-up,-52.6,143.0,3-6 6-3 6-2,completed


In [18]:
## Create chart-ready data for the selected player

player_chart_data = player_matches.copy()

player_chart_data["match_date"] = pd.to_datetime(
    player_chart_data["match_date"]
)

player_chart_data["match_label"] = (
    player_chart_data["match_date"].dt.strftime("%b %d")
    + " vs. "
    + player_chart_data["opponent"]
)

player_chart_data = player_chart_data.sort_values("match_date")

if player_chart_data.empty:
    raise ValueError(
        f"No matches were found for {SELECTED_PLAYER}. "
        "Check that the player's name is spelled exactly as it appears in usa."
    )

## First and Second Serve Performances

In [20]:
serve_long = player_chart_data.melt(
    id_vars=[
        "match_date",
        "match_label",
        "opponent",
        "tournament",
        "surface",
        "result",
    ],
    value_vars=[
        "first_serve_in_pct",
        "first_serve_win_pct",
        "second_serve_win_pct",
    ],
    var_name="serve_metric",
    value_name="percentage",
)

serve_labels = {
    "first_serve_in_pct": "First serves in",
    "first_serve_win_pct": "First-serve points won",
    "second_serve_win_pct": "Second-serve points won",
}

serve_long["serve_metric"] = (
    serve_long["serve_metric"].map(serve_labels)
)

selected_player_serve_chart = (
    alt.Chart(serve_long)
    .mark_bar()
    .encode(
        x=alt.X(
            "match_label:N",
            title="Match",
            sort=alt.EncodingSortField(
                field="match_date",
                order="ascending",
            ),
            axis=alt.Axis(labelAngle=-45),
        ),
        y=alt.Y(
            "percentage:Q",
            title="Percentage",
            axis=alt.Axis(format=".0%"),
            scale=alt.Scale(domain=[0, 1]),
        ),
        xOffset="serve_metric:N",
        color=alt.Color(
            "serve_metric:N",
            title="Serve measure",
            scale=alt.Scale(
                domain=[
                    "First serves in",
                    "First-serve points won",
                    "Second-serve points won",
                ],
                range=[
                    "#4C78A8",
                    "#59A14F",
                    "#F28E2B",
                ],
            ),
        ),
        tooltip=[
            alt.Tooltip("match_date:T", title="Date"),
            alt.Tooltip("opponent:N", title="Opponent"),
            alt.Tooltip("tournament:N", title="Tournament"),
            alt.Tooltip("surface:N", title="Surface"),
            alt.Tooltip("result:N", title="Result"),
            alt.Tooltip("serve_metric:N", title="Measure"),
            alt.Tooltip(
                "percentage:Q",
                title="Percentage",
                format=".1%",
            ),
        ],
    )
    .properties(
        width=850,
        height=400,
        title=f"{SELECTED_PLAYER}: Serve Performance by Match",
    )
)

selected_player_serve_chart

alt.Chart(...)

## Break point opportunities and conversions

In [24]:
break_point_long = player_chart_data.melt(
    id_vars=[
        "match_date",
        "match_label",
        "opponent",
        "tournament",
        "surface",
        "result",
        "bp_conversion_pct",
    ],
    value_vars=[
        "bp_opportunities",
        "bp_converted",
    ],
    var_name="break_point_metric",
    value_name="total",
)

break_point_labels = {
    "bp_opportunities": "Opportunities",
    "bp_converted": "Converted",
}

break_point_long["break_point_metric"] = (
    break_point_long["break_point_metric"].map(
        break_point_labels
    )
)

selected_player_bp_chart = (
    alt.Chart(break_point_long)
    .mark_bar()
    .encode(
        x=alt.X(
            "match_label:N",
            title="Match",
            sort=alt.EncodingSortField(
                field="match_date",
                order="ascending",
            ),
            axis=alt.Axis(labelAngle=-45),
        ),
        y=alt.Y(
            "total:Q",
            title="Break points",
        ),
        xOffset="break_point_metric:N",
        color=alt.Color(
            "break_point_metric:N",
            title="Break-point measure",
            scale=alt.Scale(
                domain=["Opportunities", "Converted"],
                range=["#9ECAE1", "#E15759"],
            ),
        ),
        tooltip=[
            alt.Tooltip("match_date:T", title="Date"),
            alt.Tooltip("opponent:N", title="Opponent"),
            alt.Tooltip("tournament:N", title="Tournament"),
            alt.Tooltip("surface:N", title="Surface"),
            alt.Tooltip("result:N", title="Result"),
            alt.Tooltip(
                "bp_conversion_pct:Q",
                title="Conversion rate",
                format=".1%",
            ),
            alt.Tooltip(
                "break_point_metric:N",
                title="Measure",
            ),
            alt.Tooltip("total:Q", title="Total"),
        ],
    )
    .properties(
        width=850,
        height=400,
        title=(
            f"{SELECTED_PLAYER}: Break-Point "
            "Opportunities and Conversions"
        ),
    )
)

selected_player_bp_chart

alt.Chart(...)

## Break point convsersion and save percentages

In [26]:
break_rate_long = player_chart_data.melt(
    id_vars=[
        "match_date",
        "match_label",
        "opponent",
        "tournament",
        "surface",
        "result",
        "bp_opportunities",
    ],
    value_vars=[
        "bp_conversion_pct",
        "bp_save_pct",
    ],
    var_name="break_metric",
    value_name="percentage",
)

break_rate_labels = {
    "bp_conversion_pct": "Break-point conversion",
    "bp_save_pct": "Break-point save",
}

break_rate_long["break_metric"] = (
    break_rate_long["break_metric"].map(
        break_rate_labels
    )
)

selected_player_break_rate_chart = (
    alt.Chart(break_rate_long)
    .mark_line(point=True, strokeWidth=3)
    .encode(
        x=alt.X(
            "match_label:N",
            title="Match",
            sort=alt.EncodingSortField(
                field="match_date",
                order="ascending",
            ),
            axis=alt.Axis(labelAngle=-45),
        ),
        y=alt.Y(
            "percentage:Q",
            title="Percentage",
            axis=alt.Axis(format=".0%"),
            scale=alt.Scale(domain=[0, 1]),
        ),
        color=alt.Color(
            "break_metric:N",
            title="Measure",
            scale=alt.Scale(
                domain=[
                    "Break-point conversion",
                    "Break-point save",
                ],
                range=["#E15759", "#4C78A8"],
            ),
        ),
        tooltip=[
            alt.Tooltip("match_date:T", title="Date"),
            alt.Tooltip("opponent:N", title="Opponent"),
            alt.Tooltip("tournament:N", title="Tournament"),
            alt.Tooltip("surface:N", title="Surface"),
            alt.Tooltip("result:N", title="Result"),
            alt.Tooltip("break_metric:N", title="Measure"),
            alt.Tooltip(
                "percentage:Q",
                title="Percentage",
                format=".1%",
            ),
            alt.Tooltip(
                "bp_opportunities:Q",
                title="Break opportunities",
            ),
        ],
    )
    .properties(
        width=850,
        height=400,
        title=(
            f"{SELECTED_PLAYER}: Break-Point "
            "Conversion and Save Rates"
        ),
    )
)

selected_player_break_rate_chart

alt.Chart(...)

## 9. Interactive Altair visualizations

All charts support interactive tooltips. The time-series chart also supports zooming and horizontal panning.

In [ ]:
# Win percentage by player (minimum 3 matches)
plot_summary = summary.query("matches >= 3").copy()

win_chart = (
    alt.Chart(plot_summary)
    .mark_bar()
    .encode(
        x=alt.X("win_pct:Q", title="Win percentage", axis=alt.Axis(format=".0%")),
        y=alt.Y("player:N", title=None, sort="-x"),
        color=alt.Color("matches:Q", title="Matches", scale=alt.Scale(scheme="viridis")),
        tooltip=[
            alt.Tooltip("player:N", title="Player"),
            alt.Tooltip("matches:Q", title="Matches"),
            alt.Tooltip("wins:Q", title="Wins"),
            alt.Tooltip("losses:Q", title="Losses"),
            alt.Tooltip("win_pct:Q", title="Win %", format=".1%"),
        ],
    )
    .properties(
        width=700,
        height=max(350, 24 * len(plot_summary)),
        title="2026 Win Percentage: U.S. Men in Grand Slams and Masters 1000",
    )
)
win_chart

In [ ]:
# Surface performance for players with at least 3 matches on a surface
surface = (usa.groupby(["player", "surface"], as_index=False)
           .agg(matches=("win", "size"), wins=("win", "sum"), win_pct=("win", "mean"))
           .query("matches >= 3"))
player_order = summary.sort_values("wins", ascending=False)["player"].tolist()

surface_base = alt.Chart(surface).encode(
    x=alt.X("surface:N", title=None),
    y=alt.Y("player:N", title=None, sort=player_order),
    tooltip=[
        alt.Tooltip("player:N", title="Player"),
        alt.Tooltip("surface:N", title="Surface"),
        alt.Tooltip("matches:Q", title="Matches"),
        alt.Tooltip("wins:Q", title="Wins"),
        alt.Tooltip("win_pct:Q", title="Win %", format=".1%"),
    ],
)
surface_chart = (
    surface_base.mark_rect()
    .encode(color=alt.Color("win_pct:Q", title="Win %", scale=alt.Scale(scheme="redyellowgreen", domain=[0, 1])))
    + surface_base.mark_text(fontSize=11).encode(
        text=alt.Text("win_pct:Q", format=".0%"),
        color=alt.condition("datum.win_pct > 0.75 || datum.win_pct < 0.25", alt.value("white"), alt.value("black")),
    )
).properties(
    width=400,
    height=max(350, 24 * surface["player"].nunique()),
    title="Win Percentage by Surface (minimum 3 matches)",
)
surface_chart

In [ ]:
# Serve effectiveness: first-serve points won vs second-serve points won
serve = (usa.groupby("player", as_index=False)
         .agg(matches=("match_key", "size"), first_serve_win_pct=("first_serve_win_pct", "mean"),
              second_serve_win_pct=("second_serve_win_pct", "mean"), win_pct=("win", "mean"))
         .query("matches >= 5").dropna())
serve["short_name"] = serve["player"].str.split().str[-1]

serve_points = alt.Chart(serve).mark_circle(opacity=.85, stroke="white", strokeWidth=1).encode(
    x=alt.X("first_serve_win_pct:Q", title="First-serve points won", axis=alt.Axis(format=".0%"), scale=alt.Scale(zero=False)),
    y=alt.Y("second_serve_win_pct:Q", title="Second-serve points won", axis=alt.Axis(format=".0%"), scale=alt.Scale(zero=False)),
    size=alt.Size("matches:Q", title="Matches", scale=alt.Scale(range=[80, 800])),
    color=alt.Color("win_pct:Q", title="Win %", scale=alt.Scale(scheme="viridis")),
    tooltip=[
        alt.Tooltip("player:N", title="Player"),
        alt.Tooltip("matches:Q", title="Matches"),
        alt.Tooltip("first_serve_win_pct:Q", title="1st serve won", format=".1%"),
        alt.Tooltip("second_serve_win_pct:Q", title="2nd serve won", format=".1%"),
        alt.Tooltip("win_pct:Q", title="Win %", format=".1%"),
    ],
)
serve_labels = alt.Chart(serve).mark_text(dx=10, dy=-8, fontSize=10).encode(
    x="first_serve_win_pct:Q", y="second_serve_win_pct:Q", text="short_name:N"
)
serve_chart = (serve_points + serve_labels).properties(
    width=700, height=450, title="Serve Effectiveness (minimum 5 matches)"
)
serve_chart

In [ ]:
# Ranking upsets
upsets = (usa.groupby("player", as_index=False)["upset_win"].sum()
          .query("upset_win > 0"))

upset_chart = (
    alt.Chart(upsets)
    .mark_bar(color="#2a6fbb")
    .encode(
        x=alt.X("upset_win:Q", title="Upset wins", axis=alt.Axis(tickMinStep=1)),
        y=alt.Y("player:N", title=None, sort="-x"),
        tooltip=[alt.Tooltip("player:N", title="Player"), alt.Tooltip("upset_win:Q", title="Upset wins")],
    )
    .properties(width=650, height=max(300, 24 * len(upsets)), title="Wins over Higher-Ranked Opponents")
)
upset_chart

In [ ]:
# Tournament results: wins per player and event
tourney = (usa.groupby(["player", "tournament"], as_index=False)
           .agg(matches=("win", "size"), wins=("win", "sum")))
player_order = summary.sort_values("wins", ascending=False)["player"].tolist()

tourney_base = alt.Chart(tourney).encode(
    x=alt.X("tournament:N", title=None, sort=None, axis=alt.Axis(labelAngle=-40)),
    y=alt.Y("player:N", title=None, sort=player_order),
    tooltip=[
        alt.Tooltip("player:N", title="Player"),
        alt.Tooltip("tournament:N", title="Tournament"),
        alt.Tooltip("matches:Q", title="Matches"),
        alt.Tooltip("wins:Q", title="Wins"),
    ],
)
tourney_chart = (
    tourney_base.mark_rect().encode(
        color=alt.Color("wins:Q", title="Match wins", scale=alt.Scale(scheme="blues"))
    )
    + tourney_base.mark_text(fontSize=10).encode(
        text=alt.Text("wins:Q", format=".0f"),
        color=alt.condition("datum.wins >= 4", alt.value("white"), alt.value("black")),
    )
).properties(
    width=700,
    height=max(400, 24 * tourney["player"].nunique()),
    title="Tournament Results: Match Wins",
)
tourney_chart

In [ ]:
# Performance over time for the leading players by total wins
leaders = summary.head(8)["player"].tolist()
trend = usa[usa["player"].isin(leaders)].sort_values(["player", "match_date", "match_key"]).copy()
trend["match_number"] = trend.groupby("player").cumcount() + 1
trend["cumulative_win_pct"] = trend.groupby("player")["win"].cumsum() / trend["match_number"]

trend_chart = (
    alt.Chart(trend)
    .mark_line(point=True, strokeWidth=2)
    .encode(
        x=alt.X("match_date:T", title="Match date"),
        y=alt.Y("cumulative_win_pct:Q", title="Cumulative win percentage", axis=alt.Axis(format=".0%"), scale=alt.Scale(domain=[0, 1])),
        color=alt.Color("player:N", title="Player"),
        tooltip=[
            alt.Tooltip("match_date:T", title="Date"),
            alt.Tooltip("player:N", title="Player"),
            alt.Tooltip("match_number:Q", title="Match number"),
            alt.Tooltip("cumulative_win_pct:Q", title="Cumulative win %", format=".1%"),
        ],
    )
    .properties(width=750, height=450, title="Cumulative Win Percentage Over 2026")
    .interactive(bind_y=False)
)
trend_chart

In [ ]:
# Deciding-set and tiebreak performance
clutch = (usa.groupby("player", as_index=False)
          .agg(matches=("win", "size"), deciding_sets=("deciding_set_played", "sum"),
               deciding_set_wins=("deciding_set_win", "sum"),
               tiebreaks=("tiebreaks_played", "sum"),
               tiebreaks_won=("player_tiebreaks_won", "sum")))
clutch["deciding_set_win_pct"] = safe_divide(clutch["deciding_set_wins"], clutch["deciding_sets"])
clutch["tiebreak_win_pct"] = safe_divide(clutch["tiebreaks_won"], clutch["tiebreaks"])
clutch_long = clutch.melt(
    id_vars=["player", "matches", "deciding_sets", "tiebreaks"],
    value_vars=["deciding_set_win_pct", "tiebreak_win_pct"],
    var_name="metric", value_name="rate",
).dropna()
clutch_long["metric"] = clutch_long["metric"].map({
    "deciding_set_win_pct": "Deciding sets", "tiebreak_win_pct": "Tiebreaks"
})

clutch_chart = (
    alt.Chart(clutch_long)
    .mark_bar()
    .encode(
        x=alt.X("rate:Q", title="Win percentage", axis=alt.Axis(format=".0%")),
        y=alt.Y("player:N", title=None, sort="-x"),
        color=alt.Color("metric:N", title="Metric"),
        yOffset="metric:N",
        tooltip=[
            alt.Tooltip("player:N", title="Player"), alt.Tooltip("metric:N", title="Metric"),
            alt.Tooltip("rate:Q", title="Win %", format=".1%"),
            alt.Tooltip("deciding_sets:Q", title="Deciding sets"),
            alt.Tooltip("tiebreaks:Q", title="Tiebreaks"),
        ],
    )
    .properties(width=650, height=max(350, 28 * clutch_long["player"].nunique()),
                title="Deciding-Set and Tiebreak Performance")
)
clutch_chart

In [ ]:
# Actual versus ranking-expected win percentage (minimum 5 matches)
expectation_plot = expectation_summary.query("matches >= 5").copy()
expectation_long = expectation_plot.melt(
    id_vars=["player", "matches", "performance_vs_expectation"],
    value_vars=["actual_win_pct", "expected_win_pct"],
    var_name="metric", value_name="win_pct",
)
expectation_long["metric"] = expectation_long["metric"].map({
    "actual_win_pct": "Actual", "expected_win_pct": "Ranking expected"
})
expectation_order = expectation_plot.sort_values("performance_vs_expectation", ascending=False)["player"].tolist()

expectation_chart = (
    alt.Chart(expectation_long)
    .mark_bar()
    .encode(
        x=alt.X("player:N", title=None, sort=expectation_order, axis=alt.Axis(labelAngle=-45)),
        xOffset="metric:N",
        y=alt.Y("win_pct:Q", title="Win percentage", axis=alt.Axis(format=".0%")),
        color=alt.Color("metric:N", title=None),
        tooltip=[
            alt.Tooltip("player:N", title="Player"), alt.Tooltip("metric:N", title="Measure"),
            alt.Tooltip("win_pct:Q", title="Win %", format=".1%"),
            alt.Tooltip("performance_vs_expectation:Q", title="Over/under expectation", format="+.1%"),
            alt.Tooltip("matches:Q", title="Matches"),
        ],
    )
    .properties(width=750, height=420,
                title="Actual vs Ranking-Expected Win Percentage (minimum 5 matches)")
)
expectation_chart

## 10. Additional validation and data-quality notes

- Missing serve statistics remain missing; they are not replaced with zero.
- A break-point percentage is missing when no relevant opportunities occurred.
- Rankings are match-time ATP rankings.
- `ranking_difference = opponent_rank - player_rank`; negative values mean the American was ranked lower.
- Ranking-based probabilities are a transparent ordinal-rank benchmark, not calibrated betting probabilities.
- Score-derived statistics exclude unparseable sets; retirements and walkovers remain explicitly flagged.
- Live data may be revised by the provider. Rerunning the notebook refreshes and deduplicates it.
- Events not yet played in 2026 do not appear until the source publishes matches.

In [ ]:
quality_report = pd.DataFrame({
    "metric": [
        "Analytical rows", "Unique qualifying matches", "U.S. players", "All-American matches",
        "Missing player rankings", "Missing opponent rankings", "Missing serve points",
        "Missing match durations", "Duplicate player-match rows"
    ],
    "value": [
        len(usa), usa["match_key"].nunique(), usa["player"].nunique(),
        big_events.eval("winner_ioc == 'USA' and loser_ioc == 'USA'").sum(),
        usa["player_rank"].isna().sum(), usa["opponent_rank"].isna().sum(),
        usa["serve_points"].isna().sum(), usa["minutes"].isna().sum(),
        usa.duplicated(["match_key", "player_id"]).sum()
    ]
})
display(quality_report)

assert usa.duplicated(["match_key", "player_id"]).sum() == 0
assert set(usa["result"].dropna().unique()) <= {"W", "L"}
assert usa["player"].notna().all()
assert usa["tournament"].isin(EXPECTED_EVENTS).all()
assert usa["year"].eq(2026).all()
print("All final assertions passed.")